In this module, we extract some insights on common subjects and their examination characteristics, based on past examination data. 
All data is sourced from https://www.vcaa.vic.edu.au/. We only use previous examination assessment reports in this notebook, as they contain the most interesting data.

In [1]:
# Resolve path
from pathlib import Path
import sys

parent_dir = str(Path().resolve().parents[0])
sys.path.insert(0, parent_dir)

In [2]:
import pandas as pd

from src.Qlassifier import api
#from nb_helpers import mcq_merged

In [3]:
# Set the scope: popular VCE subjects with large exam component
subjects = [
    "specialist_mathematics",
    "mathematical_methods",
    "physics",
    "chemistry"
]

years = range(2017, 2024) # Set timeframe

all_reports = api.process_material(subjects, years, type="report")

assert(all_reports)

In [4]:
all_reports["chemistry"]["2023"][0].head(5) # Example: 2023 examination 2 specialist math MCQ table

,Question,Correct answer,% A,% B,% C,% D,Comments
0,1.,B,8,67,8,17,"In human cells, glucose reacts with oxygen in ..."
1,2.,A,40,5,42,13,Fuel cells and galvanic cells both produce hea...
2,3.,A,70,2,26,1,The larger the number of C=C double bonds in t...
3,4.,D,14,2,5,79,The polarity of the physical electrodes does n...
4,5,D,16,38,3,42,All three statements are properties of coenzym...


There are some cool statistics & facts we can extract from this data. Below are but just a few of these which we will answer in this notebook:

- Which subjects have the "hardest" short-answer questions (and what does it mean for a question to be hard)?
- Do some subjects have a bias towards certain MCQ responses? Perhaps Chemistry exam writers are fascinated with making A) the correct answer.
- Which subjects have the highest frequency of "tricky" MCQ questions? (where the majority answer != correct answer).
 

In [5]:
all_reports["specialist_mathematics"]["2017_2"][0]

,Question,% A,% B,% C,% D,% E,% No Answer,Comments
0,1,1,8,75,4,12,0,"1 1\nx − ≤ ≤ ⇒ ∈ −∞ − ∪ ∞\n1\nx\n( , 1] [1, )"
1,2,9,30,11,12,37,1,The solve and graphing capabilities of\na CAS ...
2,3,4,4,9,47,35,0,Use of complex solve gives five\nsolutions.
3,4,14,7,11,15,53,0,None
4,5,75,6,5,10,4,0,None
5,6,6,46,10,30,8,1,2 2\ndy d e\nx\n= =\narctan( )\ny\n(\n)\ndx dx...
6,7,4,6,20,60,9,1,None
7,8,4,29,7,52,7,0,3 m\n′′ =− ≥\n≥\nfx x m\n( ) 6 2 0 when\nx
8,9,4,45,12,9,30,0,None
9,10,31,9,45,7,6,1,f x ′′( ) does not change sign at a.


In [6]:
def mcq_merged_lazy(all_reports, years):
    all_merged = {}
    for subject, subject_reports in all_reports.items():
        def df_gen():
            for report, dfs in subject_reports.items():
                if any(str(year) in report for year in years) and (("math" not in subject.lower()) or "2" in report):
                    yield dfs[0].rename(columns=lambda x: str(x) if x is not None else "Unnamed")
        
        merged = pd.concat(df_gen(), axis=0, ignore_index=True)
        all_merged[subject] = merged
    return all_merged

In [7]:
all_reports["specialist_mathematics"][]

SyntaxError: invalid syntax (1800938390.py, line 1)

In [ ]:
def standardise_columns(
    subject_reports: dict[str, list[pd.DataFrame]]
) -> dict[str, list[pd.DataFrame]]:
    def standardise_columns(
        subject_reports: dict[str, list[pd.DataFrame]]
    ) -> dict[str, list[pd.DataFrame]]:
        """
        Standardise the first DataFrame in each report list:
        - drop any '% n/a' column variants
        - normalise 'correct answer' / 'is_correct' to 'is_correct'
        - make column names lower-case and use underscores for spaces
        - ensure every first-element dataframe has exactly the same columns (adds missing cols as NA)
        """
        def _norm(col):
            if col is None:
                return None
            s = str(col).strip().lower()
            if s in ("% n/a", "% n/a ", "%n/a"):
                return None
            if s in ("correct answer", "correct_answer", "is_correct", "correctanswer"):
                return "is_correct"
            # replace spaces with underscore, collapse multiple spaces
            return "_".join(s.split())

        # First pass: normalise column names for the first element of each report, dropping % n/a
        normalised_dfs: dict[str, pd.DataFrame] = {}
        for report_key, dfs in subject_reports.items():
            if not dfs:
                continue
            df = dfs[0]
            originals = list(df.columns)
            normalized = [_norm(c) for c in originals]

            # keep first occurrence when multiple originals map to same normalized name
            seen = {}
            for idx, name in enumerate(normalized):
                if name is None:
                    continue
                if name not in seen:
                    seen[name] = idx

            # build a new dataframe with the chosen columns in the order encountered
            new_cols = {name: df.iloc[:, idx].reset_index(drop=True) for name, idx in seen.items()}
            new_df = pd.DataFrame(new_cols)

            # assign back to the first element
            subject_reports[report_key][0] = new_df
            normalised_dfs[report_key] = new_df

        if not normalised_dfs:
            return subject_reports

        # Determine canonical column order:
        # start from the first report in the dict, then append any other columns seen elsewhere
        first_report_key = next(iter(normalised_dfs))
        canonical_cols = list(normalised_dfs[first_report_key].columns)

        # collect any additional columns and append them (sorted to be deterministic)
        extra_cols = set()
        for k, df in normalised_dfs.items():
            extra_cols.update(df.columns)
        extra_cols.difference_update(canonical_cols)
        if extra_cols:
            canonical_cols += sorted(extra_cols)

        # Ensure every first-element dataframe has exactly the canonical columns (add missing as NA, reorder)
        for report_key, dfs in subject_reports.items():
            if not dfs:
                continue
            df = dfs[0]
            for col in canonical_cols:
                if col not in df.columns:
                    df[col] = pd.NA
            # reorder
            subject_reports[report_key][0] = df.loc[:, canonical_cols]

        return subject_reports

def mcq_merged(
    all_reports: dict[str, dict[str, list[pd.DataFrame]]], 
    years: list[int],
) -> dict[str, pd.DataFrame]:
    """ Returns a dictionary mapping each subject to one dataframe consisting
    of all multiple choice tables merged over the years indicated by years.
    """
    all_merged = {}
    subjects = all_reports.keys()
    for subject in subjects:
        report_types = list(all_reports[subject].keys())
        print(subject)
        print([report for report in report_types if any(str(year) in report for year in years) and ("_2" in report)])
        if "math" in subject:
            # only examination 2's have mcq sections
            dfs_to_merge = [all_reports[subject][report][0] for report in report_types \
                            if any(str(year) in report for year in years) and "2" in report]
        else: 
            dfs_to_merge = [all_reports[subject][report][0] for report in report_types \
                            if any(str(year) in report for year in years)]
        dfs_to_merge = [df.rename(columns=lambda x: str(x) if x is not None else "Unnamed") for df in dfs_to_merge]
        
        merged = pd.concat(dfs_to_merge, axis=0, ignore_index=True)
        print(merged)
        all_merged[subject] = merged
    return all_merged

mcq_merged(all_reports, years=[2020, 2021, 2022])["specialist_mathematics"]

specialist_mathematics
['2020_2', '2022_2', '2021_2']
   Question % A % B % C % D % E % N/A  \
0         1   4   8   6  70  11     0   
1         2  18  42  24  10   5     0   
2         3  68  20   4   5   2     0   
3         4   4   4  14  50  28     0   
4         5  66   4  13   9   8     0   
..      ...  ..  ..  ..  ..  ..   ...   
58       16   5  48  18  14  14   NaN   
59       17   5  20  13  11  50   NaN   
60       18   6  14  65  10   5   NaN   
61       19   4  12  24  51   8   NaN   
62       20   9  18  43  15  14   NaN   

                                             Comments  is_correct Marks    0  \
0                                                             4.0   NaN  NaN   
1                             Use transformations on.         2.0   NaN  NaN   
2                                                             1.0   NaN  NaN   
3   Options C, D and E have the correct rule but o...         5.0   NaN  NaN   
4                                                    

InvalidIndexError: Reindexing only valid with uniquely valued Index objects